# 10 -- Reproducibility Summary

Summarizes notebooks 00-09 and states, explicitly and without overstatement, this stage's actual reproducibility status against `report_current.html`.

## Blocking checklist for a genuine production backtest

| # | Requirement | Status |
|---|---|---|
| 1 | scikit-learn>=1.3.0,<2.0.0 installed | **Done** |
| 3 | requests/yfinance/lxml/pyarrow installed | **Done** |
| 4 | Real universe/sector/filing/price data acquired | **Done** -- 518 universe members, 96,852 filing rows, 4,436,726 price rows (Stage 11) |
| 5 | `DataProvenanceManifest` for that real dataset | **Done** -- `data/manifests/filing_momentum_ml/data_manifest.json` |
| 6 | Raw-data validation passing (no `FATAL`) | **Done** -- 0 fatal, 1 error, ~29,700 warnings (all disclosed, expected XBRL data-quality characteristics) |
| 7 | Feature build produces trainable rows despite non-calendar fiscal years | **Done** (this stage's own correction) -- 18,960 of 20,720 candidate attempts succeed (91.5%); every rejection is genuine data insufficiency, never a fiscal/calendar mismatch |
| 8 | Training-quarter eligibility | **Done** -- 32 of 39 candidate target quarters eligible (was 0 of every quarter before the cohort-snapshot correction) |
| 9 | Legacy `Arnold_Quant` cache independently verified, if reused | **Not attempted** -- diagnostic audit only (notebook 01); not a blocker for a fresh acquisition-based run |

## The cohort-snapshot correction

A real-data run initially showed **zero** trainable quarters across the entire real, acquired universe. Root cause (`implementation_bug`, not a missing report rule): `build_feature_observation` required a filing's own fiscal `quarter_end` to exactly equal the shared calendar cohort's date as an inclusion condition. Real issuers routinely use 52/53-week or otherwise offset fiscal years (AAPL: last-Saturday-of-month, a few days off; WMT: fiscal year ends January 31, a full month off) -- only 78.4% of real acquired filing rows are exactly calendar-aligned, and 61 of 507 real symbols have *zero* calendar-aligned quarters.

The recovered, corrected behavior (from `report_current.html`'s own `get_available_as_of` -- an *ordinal* "most recent 8 filed quarters" rule -- and the legacy `RollingMLScorer`/`data_sec.py` implementation): every ticker gets one candidate row per shared cohort from its most-recently-knowable fiscal history; exact calendar alignment only refines entry-timing precision, never gates inclusion. A "nearest calendar quarter" mapping was considered and explicitly rejected -- WMT's month-long offset would require a tolerance wide enough to risk cross-cohort ambiguity, and the report/legacy implementation never did any such mapping at all.

## Install commands (never run silently)

```bash
```

## Resume commands

```bash
atlas-quant filing-momentum validate-data --raw-root data/raw/filing_momentum_ml
atlas-quant filing-momentum run-backtest --raw-root data/raw/filing_momentum_ml \
    --manifest data/manifests/filing_momentum_ml/data_manifest.json \
    --start-quarter 2015-03-31 --end-quarter 2024-12-31 \
    --regime-gate-mode none \
    --checkpoint-root data/manifests/filing_momentum_ml
```
This strategy has **no regime gate at all** -- the report's two-layer HMM+Markov gate was removed deliberately and permanently (see `docs/reproducibility_findings.md`), so `hmmlearn` is no longer a dependency and there is no gate-mode to configure. This is a design divergence from `report_current.html`, not an environment limitation. Re-running `run-backtest` with the same `--checkpoint-root` resumes from the last completed checkpoint; a checkpoint computed under a different dataset/config identity is rejected (`BLOCKED_IDENTITY_MISMATCH`), never silently reused.

## Data-manifest schema

See `docs/data_provenance_manifest.md` for the full field-by-field schema `DataProvenanceManifest.to_dict()`/`from_dict()` reads and writes.

## Final reproducibility classification

**`NOT_REPRODUCIBLE_CONFIGURATION_MISMATCH`** (`atlas_quant.reporting.domain.ReproducibilityStatus`).

A genuine historical backtest **has actually run to completion** against real, acquired data: 518 real symbols, 2015-03-31 through 2024-12-31, 32 of 40 shared cohorts completed with a real `HistGradientBoostingClassifier` fit per quarter. Result: +778.8% cumulative return, Sharpe 1.22, Sortino 4.39, 71.9% win rate, all 32 completed quarters in primary (stock-picking) mode.

**This is not a reproduction of `report_current.html`, for two independent, disclosed reasons:**

1. The strategy implements no regime gate at all (the report's `gate_mode="both"` HMM+Markov gate was deliberately removed), and its below-`min_positions` behavior is a VOO/VTI partial-fill capital sleeve rather than the report's all-or-nothing SPY/VGT fallback. Both are permanent design divergences, not reproduction gaps.
2. SPY/VGT (the report's benchmark and fallback tickers) were never acquired -- Stage 11's acquisition only fetches S&P 500 + Nasdaq 100 *constituents*, and SPY/VGT are index/sector ETFs, not constituents of their own index. Every quarter's benchmark return is `None` as a direct, confirmed consequence. Classified `data_provenance_required` -- a real gap, not fixed in this stage.

**Do not cite this run's own return/Sharpe numbers as evidence of reproducing `report_current.html`.** They are real, not synthetic, but computed under a disabled regime gate with no benchmark comparison available. See `docs/reproducibility_findings.md` for the full detail.

## Findings (series-wide)

- The full offline production-research pipeline -- dependency gating, data provenance, legacy audit, validation, normalization, feature/label build, model/regime boundaries, backtest/performance/report orchestration, checkpointing -- is implemented, tested, and demonstrated end-to-end on synthetic data.
- Every blocking point is a clean, typed, reported state (`blocked=True` plus a reason), never a crash and never a silent fallback.

## Limitations (series-wide)

- No provider adapters exist yet for real SEC EDGAR filings, real daily prices, or a real S&P 500 + Nasdaq 100 universe snapshot -- acquiring real data is out of scope for this stage.
- Do not use any number in this notebook series as a substitute for a genuine result.